In [ ]:
📌 1. Introduction
# 5_Final_Comparison.ipynb
### Compare Perceptron, MLP, CNN, and VGG16 Models

This notebook evaluates and compares:

- Perceptron
- MLP (Backpropagation)
- CNN (Custom)
- VGG16 (Transfer Learning)

All models are tested on the same validation split (20%).  
Metrics compared:
- Accuracy
- Classification Report
- Confusion Matrix
- Training Time (if recorded)
- Summary Table + Graphs

In [ ]:
📦 2. Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
📂 3. Dataset Setup (Same 80/20 Split)
dataset_path = r"F:\\TERM 7\\CSM422 (DEEP LEARNING)\\Emotion_recognitition\\notebooks\\dataset"
img_size = (224, 224)    # For all models (VGG16 requirement)
batch_size = 32

In [ ]:
🔄 4. Load Validation Data (Common for All Models)
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# test/validation generator
val_gen = datagen.flow_from_directory(
    dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

labels = list(val_gen.class_indices.keys())

In [ ]:
🤖 5. Load All Trained Models
models = {
    "MLP": "mlp_model.h5",
    "CNN": "cnn_emotion_model.h5",
    "VGG16": "vgg16_emotion_model.h5"
    # "Perceptron": "perceptron_model.h5"  # Only if you saved it earlier
}

loaded_models = {}

for name, path in models.items():
    if os.path.exists(path):
        loaded_models[name] = load_model(path)
        print(f"Loaded: {name}")
    else:
        print(f"⚠️ Model file not found: {name} ({path})")

In [ ]:
🧪 6. Evaluate Each Model
model_results = {}

for name, model in loaded_models.items():
    print(f"\nEvaluating {name}...")
    val_gen.reset()
    pred_probs = model.predict(val_gen)
    pred_classes = np.argmax(pred_probs, axis=1)
    true_classes = val_gen.classes

    accuracy = accuracy_score(true_classes, pred_classes)

    model_results[name] = {
        "accuracy": accuracy,
        "pred_classes": pred_classes,
        "true_classes": true_classes,
        "report": classification_report(true_classes, pred_classes, target_names=labels, output_dict=True)
    }

    print(f"{name} Accuracy: {accuracy:.4f}")

In [ ]:
📊 7. Accuracy Bar Chart
names = list(model_results.keys())
acc_values = [model_results[n]["accuracy"] for n in names]

plt.figure(figsize=(10,5))
plt.bar(names, acc_values, color=["blue","green","orange","red"])
plt.title("Model Accuracy Comparison")
plt.ylabel("Accuracy")
plt.ylim(0,1)
plt.show()

In [ ]:
🔍 8. Confusion Matrix for Each Model
for name in model_results:
    cm = confusion_matrix(
        model_results[name]["true_classes"],
        model_results[name]["pred_classes"]
    )

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix - {name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.show()

In [ ]:
🧾 9. Tabular Summary
import pandas as pd

summary = {
    "Model": [],
    "Accuracy": []
}

for name in model_results:
    summary["Model"].append(name)
    summary["Accuracy"].append(model_results[name]["accuracy"])

df_summary = pd.DataFrame(summary)
df_summary

In [ ]:
📝 10. Final Analysis
# Final Analysis

### Typically:
- **Perceptron** → lowest performance  
- **MLP** → better but still limited by no spatial feature extraction  
- **CNN** → strong baseline, learns spatial patterns  
- **VGG16 (Transfer Learning)** → highest accuracy because of pretrained weights on ImageNet

### Factors Affecting Differences:
- Dataset size  
- Depth of model  
- Transfer learning advantages  
- Image resolution  
- Training time  
- Overfitting control  